# Playground — DeCo-VTON try-on (AutoMasker)

Person + cloth only (no dataset agnostic). Builds the agnostic mask with AutoMasker (DensePose + SCHP), then runs [DeCo-VTON](https://huggingface.co/levinna/DeCo-VTON).

**Prerequisites** (CUDA recommended; Detectron2 DensePose is not reliable on MPS):

```bash
.venv/bin/python -m pip install opencv-python
.venv/bin/python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'
.venv/bin/python -m pip install 'git+https://github.com/facebookresearch/detectron2.git#subdirectory=projects/DensePose'
```

CatVTON preprocess weights (`DensePose/`, `SCHP/`) are downloaded from `zhengchong/CatVTON` on first `load_automasker` call.

In [1]:
import sys
from pathlib import Path

import torch

sys.path.insert(0, str(Path("../..").resolve()))

from fashion_diffusion.paths import DATA_DIR
from fashion_diffusion.agnostic_mask import generate_agnostic_mask, load_automasker
from fashion_diffusion.decovton import (
    load_pipeline,
    prepare_inputs_from_mask,
    run_tryon,
    save_result,
    show_condition_and_mask,
    show_person_cloth_mask,
    show_result,
)

W1126 22:47:52.843000 14849 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
person_filename = "03195_00.jpg"
cloth_filename = "00009_00.jpg"
split = "train"

person_path = DATA_DIR / split / "image" / person_filename
cloth_path = DATA_DIR / split / "cloth" / cloth_filename
outputs_dir = DATA_DIR / "outputs" / "playground" / "decovton_tryon_automasker"

hf_repo = "levinna/DeCo-VTON"
hf_subfolder = "VITON-HD-512/unet"
catvton_repo = "zhengchong/CatVTON"
mask_type = "upper"
mask_device = "cuda"
tryon_device = "cuda"
dtype = torch.float16
num_inference_steps = 50

print("==== Input paths:")
print(f"\tperson_path: {person_path}")
print(f"\tcloth_path: {cloth_path}")
print(f"\thf_repo: {hf_repo}")
print(f"\thf_subfolder: {hf_subfolder}")
print(f"\tcatvton_repo: {catvton_repo}")
print(f"\tmask_type: {mask_type}")
print(f"\tmask_device: {mask_device}")
print(f"\ttryon_device: {tryon_device}")
print(f"\tnum_inference_steps: {num_inference_steps}")

print("\n==== Output paths:")
print(f"\toutputs_dir: {outputs_dir}")

==== Input paths:
	person_path: /Users/luigidamico/NoCloudStuff/fashion-diffusion/data/HR-VITON/train/image/03195_00.jpg
	cloth_path: /Users/luigidamico/NoCloudStuff/fashion-diffusion/data/HR-VITON/train/cloth/00009_00.jpg
	hf_repo: levinna/DeCo-VTON
	hf_subfolder: VITON-HD-512/unet
	catvton_repo: zhengchong/CatVTON
	mask_type: upper
	mask_device: cuda
	tryon_device: cuda
	num_inference_steps: 50

==== Output paths:
	outputs_dir: /Users/luigidamico/NoCloudStuff/fashion-diffusion/data/HR-VITON/outputs/playground/decovton_tryon_automasker


In [3]:
automasker = load_automasker(catvton_repo, device=mask_device)
mask_pil = generate_agnostic_mask(automasker, person_path, mask_type=mask_type)
show_person_cloth_mask(person_path, cloth_path, mask_pil)

/Users/luigidamico/Documents/GitHub/fashion-diffusion/.venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
objc[14849]: Class AVFFrameReceiver is implemented in both /Users/luigidamico/Documents/GitHub/fashion-diffusion/.venv/lib/python3.14/site-packages/cv2/.dylibs/libavdevice.63.1.102.dylib (0x1223f8328) and /Users/luigidamico/Documents/GitHub/fashion-diffusion/.venv/lib/python3.14/site-packages/av/.dylibs/libavdevice.63.1.102.dylib (0x1227f83a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[14849]: Class AVFAudioReceiver is implemented in both /Users/luigidamico/Documents/GitHub/fashion-diffusion/.venv/lib/python3.14/site-packages/cv2/.dylibs/libavdevice.63.1.102.dylib (0x1223f8378) and /Users/luigidamico/Documents/GitHub/fashion-diffusion/.venv/lib/py

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
condition, mask = prepare_inputs_from_mask(person_path, cloth_path, mask_pil)
show_condition_and_mask(condition, mask)

In [ ]:
pipe = load_pipeline(hf_repo, hf_subfolder, tryon_device, dtype)
result = run_tryon(pipe, condition, mask, num_inference_steps=num_inference_steps)
out_path = save_result(result, outputs_dir / f"{person_path.stem}__{cloth_path.stem}.png")
print(f"saved: {out_path}")
show_result(person_path, cloth_path, result)